In [ ]:
import numpy as np

# Data Generation

In [ ]:
np.random.seed(42)

# fail_study  = np.random.normal(loc=3, scale=1.5, size=50)
fail_study  = np.random.normal(loc=3, scale=1.5, size=80)
fail_attend = np.random.normal(loc=55, scale=12, size=50)
# pass_study  = np.random.normal(loc=8, scale=2, size=50)
# pass_attend = np.random.normal(loc=80, scale=10, size=50)
# pass_study  = np.random.normal(loc=5, scale=2, size=50)
pass_study  = np.random.normal(loc=8, scale=2, size=20)
pass_attend = np.random.normal(loc=68, scale=10, size=50)

X = np.column_stack([
    np.concatenate([fail_study, pass_study]),
    np.concatenate([fail_attend, pass_attend])
])
y = np.concatenate([np.zeros(50), np.ones(50)]).astype(int)

shuffle_idx = np.random.permutation(len(y))
X, y = X[shuffle_idx], y[shuffle_idx]

# Splitting the dataset

In [ ]:
split = int(0.8 * len(y))
X_train_raw, X_test_raw = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

class Normalizer:
    def fit(self, data):
        self.mean = np.mean(data, axis=0)
        self.std  = np.std(data, axis=0)
        return self
    def transform(self, data):
        return (data - self.mean) / self.std

normalizer = Normalizer().fit(X_train_raw)     
X_train = normalizer.transform(X_train_raw)
X_test  = normalizer.transform(X_test_raw)

print(f"Training: {len(y_train)} | Test: {len(y_test)}\n")

# Higher LR + Shuffling and Sanity Checks


In [ ]:
def step_function(x):
    return (x >= 0).astype(int)

class Perceptron:
    def __init__(self, input_size, learning_rate=0.1):
        self.weights = np.zeros(input_size)  # zero init is fine
        self.bias = 0.0
        self.lr = learning_rate

    def predict(self, X):
        return step_function(X @ self.weights + self.bias)

    def train(self, X, y, epochs=100, verbose=True):
        for epoch in range(epochs):
            # FIX 3: shuffle training order each epoch
            idx = np.random.permutation(len(y))
            X_shuf, y_shuf = X[idx], y[idx]

            errors = 0
            for xi, target in zip(X_shuf, y_shuf):
                pred = 1 if (xi @ self.weights + self.bias) >= 0 else 0
                error = target - pred
                self.weights += self.lr * error * xi
                self.bias    += self.lr * error
                errors       += int(error != 0)

            if verbose and (epoch + 1) % 10 == 0:
                acc = 1.0 - errors / len(y)
                print(f"  Epoch {epoch+1:>3} | Errors: {errors:>2} | Acc: {acc:.1%}")

            if errors == 0:
                if verbose:
                    print(f"\n  Converged at epoch {epoch + 1}!")
                break

p = Perceptron(input_size=X_train.shape[1], learning_rate=0.1)
p.train(X_train, y_train, epochs=100)

In [ ]:
print(f"\n  Learned weights: {p.weights}")
print(f"  Learned bias:   {p.bias:.4f}")

if p.weights[1] < 0:
    print("  WARNING: Attendance weight is negative — something is wrong!")
else:
    print("  Both weights positive — model makes intuitive sense.")

# Evaluate

In [ ]:
y_pred = p.predict(X_test)
acc = np.mean(y_pred == y_test)

tp = np.sum((y_pred == 1) & (y_test == 1))
tn = np.sum((y_pred == 0) & (y_test == 0))
fp = np.sum((y_pred == 1) & (y_test == 0))
fn = np.sum((y_pred == 0) & (y_test == 1))

print(f"\n  Test Accuracy: {acc:.1%}")
print(f"  Confusion: TP={tp} TN={tn} FP={fp} FN={fn}")
print(f"  Precision: {tp/(tp+fp):.1%}" if tp+fp > 0 else "  Precision: N/A")
print(f"  Recall:    {tp/(tp+fn):.1%}" if tp+fn > 0 else "  Recall: N/A")


# Predicting new students 

In [ ]:
new_students = np.array([
    [5,  65],
    [10, 92],
    [2,  40],
    [7,  78],
    [4,  70],
    [9,  50],
])

new_normalized = normalizer.transform(new_students)
predictions = p.predict(new_normalized)

print(f"\n  {'Study':>8} {'Attend':>8} {'Prediction':>12}")
print(f"  {'─'*8} {'─'*8} {'─'*12}")
for student, pred in zip(new_students, predictions):
    label = "PASS" if pred == 1 else "FAIL"
    print(f"  {student[0]:>8.1f} {student[1]:>7.0f}% {label:>12}")
